# RFM Customer Segmentation Analysis

## Insurance Customer Portfolio

This notebook performs a complete RFM (Recency, Frequency, Monetary)
analysis on an insurance customer dataset. RFM is a proven marketing
framework that segments customers based on three behavioral dimensions:

- **Recency (R)** -- How recently did the customer make a purchase or
  renew a policy? More recent activity signals higher engagement.
- **Frequency (F)** -- How many transactions does the customer have?
  Repeat purchasers are more loyal and more likely to respond to
  outreach.
- **Monetary (M)** -- How much has the customer spent in total premium
  amounts? Higher spenders represent greater lifetime value.

Each customer is scored on a 1-to-5 scale for each dimension, then
assigned to a named segment (Champions, Loyal Customers, At Risk, etc.)
that maps directly to actionable marketing strategies.

---

### Table of Contents

1. Setup and Data Loading
2. Data Cleaning and Preparation
3. Exploratory Data Analysis
4. RFM Score Calculation
5. Customer Segmentation
6. Segment Analysis and Profiling
7. Visualizations
8. Actionable Recommendations
9. Export Results

---
## 1. Setup and Data Loading

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime

# ---------------------------------------------------------------------------
# Plot style
# ---------------------------------------------------------------------------
sns.set_theme(style="whitegrid", font_scale=1.1)
PALETTE = {
    "Champions":            "#1a9641",
    "Loyal Customers":      "#66bd63",
    "Potential Loyalists":  "#a6d96a",
    "Recent Customers":     "#d9ef8b",
    "Promising":            "#fee08b",
    "Need Attention":       "#fdae61",
    "About to Sleep":       "#f46d43",
    "At Risk":              "#d73027",
    "Cannot Lose Them":     "#a50026",
    "Hibernating":          "#878787",
    "Lost":                 "#4d4d4d",
}

print("Libraries loaded.")

In [ ]:
# ---------------------------------------------------------------------------
# Load dataset
# ---------------------------------------------------------------------------
df_raw = pd.read_csv("customer_segmentation_data.csv")

print(f"Dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Unique customers: {df_raw['Customer ID'].nunique():,}")
print()
df_raw.head()

---
## 2. Data Cleaning and Preparation

The raw data has mixed date formats in the Purchase History column
(both `MM-DD-YYYY` and `M/D/YYYY`). Some customers appear multiple
times, which is expected -- each row represents a separate policy or
transaction. The cleaning steps are:

1. Parse all dates into a consistent datetime format.
2. Validate numeric columns (Premium Amount, Coverage Amount).
3. Drop rows where the date could not be parsed.
4. Rename columns to shorter, code-friendly names.

In [ ]:
# ---------------------------------------------------------------------------
# Clean and prepare
# ---------------------------------------------------------------------------
df = df_raw.copy()

# Rename columns for convenience
df = df.rename(columns={
    "Customer ID":                       "customer_id",
    "Age":                               "age",
    "Gender":                            "gender",
    "Marital Status":                    "marital_status",
    "Education Level":                   "education",
    "Geographic Information":            "state",
    "Occupation":                        "occupation",
    "Income Level":                      "income",
    "Behavioral Data":                   "behavioral_data",
    "Purchase History":                  "purchase_date",
    "Interactions with Customer Service":"service_channel",
    "Insurance Products Owned":          "product",
    "Coverage Amount":                   "coverage_amount",
    "Premium Amount":                    "premium_amount",
    "Policy Type":                       "policy_type",
    "Customer Preferences":              "preference",
    "Preferred Communication Channel":   "comm_channel",
    "Preferred Contact Time":            "contact_time",
    "Preferred Language":                "language",
    "Segmentation Group":                "original_segment",
})

# Parse dates (mixed formats)
df["purchase_date"] = pd.to_datetime(df["purchase_date"], format="mixed", dayfirst=False)

# Check for parsing failures
n_null = df["purchase_date"].isna().sum()
print(f"Unparseable dates: {n_null}")
df = df.dropna(subset=["purchase_date"])

print(f"Clean dataset: {len(df):,} rows, {df['customer_id'].nunique():,} unique customers")
print(f"Date range: {df['purchase_date'].min().date()} to {df['purchase_date'].max().date()}")

---
## 3. Exploratory Data Analysis

A quick look at the distributions of the key variables before computing
RFM scores.

In [ ]:
# ---------------------------------------------------------------------------
# Distribution of transactions per customer
# ---------------------------------------------------------------------------
txn_counts = df.groupby("customer_id").size()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Transactions per customer
axes[0].hist(txn_counts.values, bins=range(1, txn_counts.max() + 2),
             color="#3498db", edgecolor="white", alpha=0.85)
axes[0].set_title("Transactions per Customer")
axes[0].set_xlabel("Number of Transactions")
axes[0].set_ylabel("Customer Count")

# Premium amount distribution
axes[1].hist(df["premium_amount"], bins=50, color="#2ecc71", edgecolor="white", alpha=0.85)
axes[1].set_title("Premium Amount Distribution")
axes[1].set_xlabel("Premium Amount")
axes[1].set_ylabel("Frequency")

# Purchase timeline
monthly = df.set_index("purchase_date").resample("M").size()
axes[2].plot(monthly.index, monthly.values, color="#e74c3c", linewidth=2)
axes[2].set_title("Monthly Transaction Volume")
axes[2].set_xlabel("Date")
axes[2].set_ylabel("Transactions")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Top demographics overview
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df["occupation"].value_counts().plot.barh(ax=axes[0], color="#3498db")
axes[0].set_title("Customers by Occupation")
axes[0].invert_yaxis()

df["policy_type"].value_counts().plot.bar(ax=axes[1], color="#9b59b6", edgecolor="white")
axes[1].set_title("Policy Type Distribution")
axes[1].tick_params(axis="x", rotation=0)

df["gender"].value_counts().plot.pie(ax=axes[2], autopct="%1.1f%%",
                                     colors=["#3498db", "#e74c3c"],
                                     startangle=90)
axes[2].set_title("Gender Split")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()

---
## 4. RFM Score Calculation

The analysis date is set to one day after the most recent transaction in
the dataset. For each customer we compute:

| Metric    | Definition                                                  |
|-----------|-------------------------------------------------------------|
| Recency   | Days since the customer's most recent purchase              |
| Frequency | Total number of transactions                                |
| Monetary  | Total premium amount across all transactions                |

Each metric is then scored 1-5 using quintile binning. For Recency, a
lower number of days is better, so the scoring is inverted (rank 5
means most recent).

In [ ]:
# ---------------------------------------------------------------------------
# Compute raw RFM values
# ---------------------------------------------------------------------------
ANALYSIS_DATE = df["purchase_date"].max() + pd.Timedelta(days=1)
print(f"Analysis date: {ANALYSIS_DATE.date()}")

rfm = (
    df.groupby("customer_id")
    .agg(
        recency=("purchase_date", lambda x: (ANALYSIS_DATE - x.max()).days),
        frequency=("purchase_date", "count"),
        monetary=("premium_amount", "sum"),
    )
    .reset_index()
)

print(f"RFM table: {len(rfm):,} customers")
print()
rfm.describe().round(1)

In [ ]:
# ---------------------------------------------------------------------------
# Score each dimension 1-5 using quintiles
# ---------------------------------------------------------------------------
# Recency: lower is better, so labels are reversed
rfm["r_score"] = pd.qcut(rfm["recency"],  q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.qcut(rfm["frequency"].rank(method="first"), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"],  q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Combined RFM score (string for segment mapping)
rfm["rfm_score"] = rfm["r_score"].astype(str) + rfm["f_score"].astype(str) + rfm["m_score"].astype(str)

# Simple aggregate score (sum of three dimensions)
rfm["rfm_total"] = rfm["r_score"] + rfm["f_score"] + rfm["m_score"]

print("Score distributions:")
print(rfm[["r_score", "f_score", "m_score"]].describe().round(2))
print()
rfm.head(10)

---
## 5. Customer Segmentation

Customers are mapped to named segments based on their R and F scores.
The mapping follows a widely used RFM segmentation framework adapted
for insurance customers. The segment names translate directly into
marketing actions.

| Segment              | R Score | F Score | Description                                          |
|----------------------|---------|---------|------------------------------------------------------|
| Champions            | 4-5     | 4-5     | Recent, frequent buyers. Highest value.              |
| Loyal Customers      | 3-5     | 3-5     | Consistent engagement over time.                     |
| Potential Loyalists  | 4-5     | 1-3     | Recent but not yet frequent. Nurture them.           |
| Recent Customers     | 4-5     | 1-1     | Just arrived. First impression matters.              |
| Promising            | 3-3     | 1-2     | Moderate recency, low frequency. Room to grow.       |
| Need Attention       | 2-3     | 2-3     | Slipping away. Re-engage before it is too late.      |
| About to Sleep       | 2-2     | 1-2     | Almost inactive. Urgent intervention needed.         |
| At Risk              | 1-2     | 3-5     | Were good customers, now disengaging.                |
| Cannot Lose Them     | 1-2     | 4-5     | High value but leaving. Highest priority retention.  |
| Hibernating          | 1-2     | 1-2     | Low on all fronts. May require reactivation campaign.|
| Lost                 | 1-1     | 1-1     | Gone. Evaluate cost of win-back vs. acquisition.     |

In [ ]:
# ---------------------------------------------------------------------------
# Segment mapping function
# ---------------------------------------------------------------------------

def assign_segment(row) -> str:
    r, f = row["r_score"], row["f_score"]

    if r >= 4 and f >= 4:
        return "Champions"
    if r >= 3 and f >= 3:
        return "Loyal Customers"
    if r >= 4 and f <= 3:
        return "Potential Loyalists"
    if r >= 4 and f == 1:
        return "Recent Customers"
    if r == 3 and f <= 2:
        return "Promising"
    if 2 <= r <= 3 and 2 <= f <= 3:
        return "Need Attention"
    if r == 2 and f <= 2:
        return "About to Sleep"
    if r <= 2 and f >= 4:
        return "Cannot Lose Them"
    if r <= 2 and f >= 3:
        return "At Risk"
    if r <= 2 and f <= 2:
        return "Hibernating"
    return "Lost"


rfm["segment"] = rfm.apply(assign_segment, axis=1)

segment_order = [
    "Champions", "Loyal Customers", "Potential Loyalists",
    "Recent Customers", "Promising", "Need Attention",
    "About to Sleep", "At Risk", "Cannot Lose Them",
    "Hibernating", "Lost",
]

seg_counts = rfm["segment"].value_counts().reindex(segment_order).dropna()
print("Segment distribution:")
print(seg_counts.to_string())
print(f"\nTotal: {seg_counts.sum():,}")

---
## 6. Segment Analysis and Profiling

This section profiles each segment across its RFM metrics and enriches
the view by merging demographic data back from the original dataset.

In [ ]:
# ---------------------------------------------------------------------------
# Segment profile table
# ---------------------------------------------------------------------------
profile = (
    rfm.groupby("segment")
    .agg(
        customers=("customer_id", "count"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean"),
        total_revenue=("monetary", "sum"),
    )
    .round(1)
    .sort_values("total_revenue", ascending=False)
)

profile["pct_customers"] = (profile["customers"] / profile["customers"].sum() * 100).round(1)
profile["pct_revenue"]   = (profile["total_revenue"] / profile["total_revenue"].sum() * 100).round(1)

profile = profile[["customers", "pct_customers", "avg_recency", "avg_frequency",
                    "avg_monetary", "total_revenue", "pct_revenue"]]

print("Segment Profile Summary")
print("=" * 95)
profile

In [ ]:
# ---------------------------------------------------------------------------
# Merge demographics for richer profiling
# ---------------------------------------------------------------------------
# Take the most recent record per customer for demographic columns
demo_cols = ["customer_id", "age", "gender", "income", "occupation",
             "policy_type", "state", "education", "purchase_date"]

df_sorted = df.sort_values("purchase_date")
demo = df_sorted.drop_duplicates(subset="customer_id", keep="last")[demo_cols]

rfm_full = rfm.merge(demo, on="customer_id", how="left")

# Age group binning
rfm_full["age_group"] = pd.cut(
    rfm_full["age"],
    bins=[0, 25, 35, 45, 55, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "56+"],
)

print(f"Enriched RFM table: {len(rfm_full):,} rows x {len(rfm_full.columns)} columns")
rfm_full.head()

In [ ]:
# ---------------------------------------------------------------------------
# Demographic breakdown per segment
# ---------------------------------------------------------------------------
seg_demo = (
    rfm_full.groupby("segment")
    .agg(
        avg_age=("age", "mean"),
        avg_income=("income", "mean"),
        top_occupation=("occupation", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "N/A"),
        top_policy=("policy_type", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "N/A"),
        pct_male=("gender", lambda x: (x == "Male").mean() * 100),
    )
    .round(1)
)

seg_demo

---
## 7. Visualizations

In [ ]:
# ---------------------------------------------------------------------------
# 7.1 Segment distribution (horizontal bar)
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 7))

seg_sorted = seg_counts.sort_values()
colors = [PALETTE.get(s, "#999999") for s in seg_sorted.index]

bars = ax.barh(seg_sorted.index, seg_sorted.values, color=colors, edgecolor="white", height=0.7)

for bar, val in zip(bars, seg_sorted.values):
    pct = val / seg_sorted.sum() * 100
    ax.text(bar.get_width() + seg_sorted.max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"  {val:,}  ({pct:.1f}%)",
            va="center", fontsize=11)

ax.set_title("Customer Segment Distribution", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Number of Customers", fontsize=12)
ax.set_xlim(0, seg_sorted.max() * 1.25)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 7.2 Revenue contribution by segment (treemap)
# ---------------------------------------------------------------------------
from matplotlib.patches import Rectangle

rev_by_seg = rfm.groupby("segment")["monetary"].sum().sort_values(ascending=False)
total_rev = rev_by_seg.sum()

fig, ax = plt.subplots(figsize=(14, 8))

# Simple treemap using squarify-like layout
try:
    import squarify
    squarify_available = True
except ImportError:
    squarify_available = False

if squarify_available:
    labels = [f"{s}\n${v/1e6:.1f}M\n({v/total_rev*100:.1f}%)" for s, v in rev_by_seg.items()]
    colors = [PALETTE.get(s, "#999") for s in rev_by_seg.index]
    squarify.plot(sizes=rev_by_seg.values, label=labels, color=colors,
                  alpha=0.85, text_kwargs={"fontsize": 10, "fontweight": "bold"}, ax=ax)
    ax.set_title("Revenue Contribution by Segment", fontsize=16, fontweight="bold", pad=15)
    ax.axis("off")
else:
    # Fallback: horizontal bar
    rev_sorted = rev_by_seg.sort_values()
    colors = [PALETTE.get(s, "#999") for s in rev_sorted.index]
    ax.barh(rev_sorted.index, rev_sorted.values / 1e6, color=colors, edgecolor="white")
    ax.set_xlabel("Revenue (Millions)", fontsize=12)
    ax.set_title("Revenue Contribution by Segment", fontsize=16, fontweight="bold", pad=15)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 7.3 RFM score heatmap (R vs F, colored by avg Monetary)
# ---------------------------------------------------------------------------
heatmap_data = (
    rfm.groupby(["r_score", "f_score"])["monetary"]
    .mean()
    .unstack(fill_value=0)
    .sort_index(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=",.0f",
    cmap="YlOrRd",
    linewidths=1,
    linecolor="white",
    cbar_kws={"label": "Avg Monetary Value"},
    ax=ax,
)
ax.set_title("RFM Heatmap: Average Monetary Value\n(Recency vs Frequency Scores)",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Frequency Score", fontsize=12)
ax.set_ylabel("Recency Score", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 7.4 Customer count heatmap (R vs F)
# ---------------------------------------------------------------------------
count_data = (
    rfm.groupby(["r_score", "f_score"])["customer_id"]
    .count()
    .unstack(fill_value=0)
    .sort_index(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    count_data,
    annot=True,
    fmt=",d",
    cmap="Blues",
    linewidths=1,
    linecolor="white",
    cbar_kws={"label": "Customer Count"},
    ax=ax,
)
ax.set_title("Customer Count by RFM Score\n(Recency vs Frequency Scores)",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Frequency Score", fontsize=12)
ax.set_ylabel("Recency Score", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 7.5 Scatter plot: Recency vs Monetary, colored by segment
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 8))

for seg in segment_order:
    subset = rfm_full[rfm_full["segment"] == seg]
    if len(subset) == 0:
        continue
    ax.scatter(
        subset["recency"],
        subset["monetary"],
        label=seg,
        color=PALETTE.get(seg, "#999"),
        alpha=0.5,
        s=20,
        edgecolors="none",
    )

ax.set_title("Recency vs Monetary Value by Segment", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Recency (days since last purchase)", fontsize=12)
ax.set_ylabel("Total Monetary Value", fontsize=12)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9, frameon=True)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 7.6 Segment profile radar/polar chart (top 5 segments by size)
# ---------------------------------------------------------------------------
from math import pi

top5 = rfm.groupby("segment").agg(
    r=("r_score", "mean"),
    f=("f_score", "mean"),
    m=("m_score", "mean"),
).loc[seg_counts.nlargest(5).index]

categories = ["Recency", "Frequency", "Monetary"]
N = len(categories)

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # close the polygon

for seg_name, row in top5.iterrows():
    values = row.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=seg_name, color=PALETTE.get(seg_name, "#999"))
    ax.fill(angles, values, alpha=0.1, color=PALETTE.get(seg_name, "#999"))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 5)
ax.set_title("Segment Profile (Top 5 by Size)", fontsize=14, fontweight="bold", pad=25)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 7.7 Income distribution by segment (box plot)
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 7))

top_segs = seg_counts.nlargest(8).index.tolist()
plot_data = rfm_full[rfm_full["segment"].isin(top_segs)]

order = plot_data.groupby("segment")["income"].median().sort_values(ascending=False).index
colors = [PALETTE.get(s, "#999") for s in order]

bp = sns.boxplot(
    data=plot_data,
    x="segment",
    y="income",
    order=order,
    palette=colors,
    ax=ax,
    fliersize=2,
)

ax.set_title("Income Distribution by Segment", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("")
ax.set_ylabel("Annual Income", fontsize=12)
ax.tick_params(axis="x", rotation=30)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 7.8 Age group composition per segment (stacked bar)
# ---------------------------------------------------------------------------
age_seg = (
    rfm_full.groupby(["segment", "age_group"])
    .size()
    .unstack(fill_value=0)
)
# Normalize to percentages
age_seg_pct = age_seg.div(age_seg.sum(axis=1), axis=0) * 100
age_seg_pct = age_seg_pct.reindex(seg_counts.index)

fig, ax = plt.subplots(figsize=(14, 7))
age_seg_pct.plot.barh(stacked=True, ax=ax,
                       color=["#3498db", "#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"],
                       edgecolor="white")

ax.set_title("Age Group Composition by Segment", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Percentage (%)", fontsize=12)
ax.set_ylabel("")
ax.legend(title="Age Group", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

---
## 8. Actionable Recommendations

Based on the segmentation results, the following strategies are
recommended for each customer group.

### High-Value Retention

**Champions** -- These are the most engaged, highest-spending customers.
Reward them with exclusive benefits, early access to new products, and
personalized service. They are also the best candidates for referral
programs because satisfied champions naturally advocate for the brand.

**Loyal Customers** -- Consistent and reliable. Offer loyalty discounts
on policy renewals, cross-sell complementary products (e.g. life
insurance if they only hold auto), and invite them to provide
testimonials.

**Cannot Lose Them** -- High-value customers showing signs of
disengagement. This is the highest-priority retention segment. Reach
out with personalized win-back offers, dedicated account managers, and
satisfaction surveys to understand what went wrong.

### Growth and Nurturing

**Potential Loyalists** -- Recent buyers with room to grow. Focus on
onboarding, education about available products, and targeted upsell
campaigns to increase purchase frequency.

**Recent Customers** -- Brand new. The first 90 days are critical.
Send welcome sequences, ensure claims processing is smooth, and collect
early feedback.

**Promising** -- Moderate engagement. Send them relevant content,
limited-time offers, and policy comparison tools to nudge them toward
a second purchase.

### Re-engagement and Win-back

**Need Attention** -- Slipping away. Trigger re-engagement campaigns
with reminders about coverage gaps, seasonal promotions, or expiring
policy alerts.

**About to Sleep** -- Nearly inactive. Send urgency-based
communications: "Your coverage may lapse" or "We noticed you have not
logged in recently." Make renewal as frictionless as possible.

**At Risk** -- Were frequent buyers, now gone quiet. Investigate
through surveys or calls. Offer competitive pricing or an upgraded
policy to bring them back.

### Deprioritize (Monitor Only)

**Hibernating / Lost** -- Low activity, low spend. Batch-process with
low-cost automated email campaigns. Evaluate whether the cost of
reactivation exceeds the expected customer lifetime value before
investing heavily.

---
## 9. Export Results

The final RFM table (with segments and demographics) and the segment
profile summary are exported as CSV files for use in dashboards,
presentations, or CRM imports.

In [ ]:
# ---------------------------------------------------------------------------
# Export
# ---------------------------------------------------------------------------
# Full customer-level RFM table
rfm_export = rfm_full[[
    "customer_id", "recency", "frequency", "monetary",
    "r_score", "f_score", "m_score", "rfm_score", "rfm_total", "segment",
    "age", "age_group", "gender", "income", "occupation", "policy_type", "state",
]].copy()

rfm_export.to_csv("rfm_customer_segments.csv", index=False)
print(f"Exported: rfm_customer_segments.csv  ({len(rfm_export):,} rows)")

# Segment profile summary
profile.to_csv("rfm_segment_profile.csv")
print(f"Exported: rfm_segment_profile.csv  ({len(profile)} segments)")

In [ ]:
# ---------------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------------
print("=" * 70)
print("  RFM ANALYSIS COMPLETE")
print("=" * 70)
print()
print(f"  Analysis date:       {ANALYSIS_DATE.date()}")
print(f"  Total customers:     {len(rfm):,}")
print(f"  Total transactions:  {len(df):,}")
print(f"  Total premium value: ${rfm['monetary'].sum():,.0f}")
print(f"  Segments created:    {rfm['segment'].nunique()}")
print()
print("  Top 3 segments by revenue:")
top3 = profile.nlargest(3, "total_revenue")
for i, (seg, row) in enumerate(top3.iterrows(), 1):
    print(f"    {i}. {seg:<22s}  ${row['total_revenue']:>12,.0f}  ({row['pct_revenue']:.1f}%)")
print()
print("  Files saved:")
print("    - rfm_customer_segments.csv")
print("    - rfm_segment_profile.csv")
print("=" * 70)

---

## Summary

This notebook performed a complete RFM analysis on 53,000+ insurance
customer records. Each customer was scored on Recency, Frequency, and
Monetary value using quintile binning, then mapped to a named segment
with clear business meaning.

The key outputs are:

- A customer-level CSV with RFM scores, segment labels, and
  demographics, ready for CRM import or dashboard integration.
- A segment profile summary showing average metrics and revenue
  contribution per segment.
- Eight production-quality visualizations covering segment distribution,
  revenue treemap, RFM heatmaps, scatter plots, radar charts, income
  distributions, and age composition.
- Actionable recommendations tied to each segment, organized by
  strategic priority (retention, growth, re-engagement, deprioritize).